# Item 5 — Plan vs. Executed: offline analysis of the rollout recorder

Loads one `.npz` produced by `franka_v2.py --log_rollouts`, constructs the three
**"plan" candidates**, and plots them against the executed path.

**Recorder format** (keys in the npz):
- `rollouts` — `(R, H, K, 3)` rollout EE positions, env-origin-corrected; `buffer[t]` = state at **end** of knot *t*
- `weights` — `(R, K)` normalized softmax weights; **sample 0 = pinned zero-noise `U_nom`** (if `--pin_nominal`)
- `exec_path` — `(S, 3)` executed EE, same frame, every control step
- `replan_steps` — `(R,)` `step_count` at each `plan()` call (integer join key)

**Decisions still owed (Jerry):** blank 5 — which candidate is "the plan"; blank 6 — deviation summary + threshold.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

# ---- constants that mirror the sim (change here if the sim changes) ----
DT = 0.01                # physics step (s)
STEPS_PER_KNOT = 2       # decimation: knots are 0.02 s apart
KNOTS_PER_PLAN = 16      # H
STEPS_PER_PLAN = 4       # replan every 4 control steps (verified: diff(replan_steps) == 4)

In [ ]:
# ---- load ----
# Forward slashes: valid on Windows, immune to escape-sequence bugs (\f == form feed!)
npz_file_path = "logs/debugging_logs/franka_v2_200_unknown_L1_on_pB_s1_20260727_222443_tg12_ts17_te30_pin_t_pin.npz"
print("cwd:", os.getcwd())
print("exists:", os.path.exists(npz_file_path))

d = np.load(npz_file_path)
rollouts     = d["rollouts"]      # (R, H, K, 3)
weights      = d["weights"]       # (R, K)
exec_path    = d["exec_path"]     # (S, 3)
replan_steps = d["replan_steps"]  # (R,)
R, H, K, _ = rollouts.shape
print(f"R={R} replans, H={H} knots, K={K} samples, {len(exec_path)} exec samples")

In [ ]:
# ---- timelines ----
exec_t = np.arange(len(exec_path)) * DT
# Capture convention (hint-1, verified): buffer[t] = state at END of knot t,
# so knot j of plan k lands (j+1) knot-durations after the plan's birth step.
plan_t = replan_steps[:, None] * DT + np.arange(1, H + 1) * STEPS_PER_KNOT * DT
print("plan 0 spans", plan_t[0, 0], "to", plan_t[0, -1], "s  (horizon = 0.32 s)")

## The three candidates

| | candidate | what it is | caveat |
|---|---|---|---|
| (a) | **elite fan** | the `n_elite` highest-weight sampled paths | 200 real paths, none singly "the" plan — best used as an uncertainty band |
| (b) | **weighted mean** | softmax-weighted average path | **not dynamically feasible** — an average of paths is not a path any control produces |
| (c) | **committed plan** | trajectory of the zero-noise `U_nom` (pinned sample 0) | one replan "stale": at replan *k* it re-rolls the plan committed at *k−1* from the current true state |

In [ ]:
def committed_plan():
    """(c) the trajectory of the controls the planner actually commits to."""
    return rollouts[:, :, 0, :]        # the pin's dividend: sample 0 = zero-noise U_nom

def weighted_mean_plan():
    """(b) softmax-weighted average path -> (R, H, 3).
    einsum: 'rk' (weights) x 'rhkc' (rollouts), summed over shared k -> 'rhc'."""
    return np.einsum("rk,rhkc->rhc", weights, rollouts)

def elite_fan(n_elite=10):
    """(a) top-weight rollout paths per replan -> (R, H, n_elite, 3).
    take_along_axis: idx[:, None, :, None] aligns the (R, n_elite) indices with
    axis 2 of the 4-D rollout array (the K axis), broadcasting over H and xyz."""
    idx = np.argsort(weights, axis=1)[:, -n_elite:]              # (R, n_elite)
    return np.take_along_axis(rollouts, idx[:, None, :, None], axis=2)

cp = committed_plan()
wm = weighted_mean_plan()
ef = elite_fan(10)   # DECISION(Jerry): fixed 10, or 'samples covering 50% of weight'?
print("committed", cp.shape, "| weighted mean", wm.shape, "| elite fan", ef.shape)

In [ ]:
# ---- verification: at the most weight-concentrated replan, the mean must ~equal the dominant sample ----
kc = weights.max(axis=1).argmax()
i_dom = weights[kc].argmax()
gap = np.abs(wm[kc] - rollouts[kc, :, i_dom]).max()
print(f"replan {kc}: dominant weight {weights[kc, i_dom]:.3f}, mean-vs-dominant gap {gap:.4f} m")
assert gap < 0.05, "einsum axis check failed — an axis is crossed"

# pin sanity: sample 0 should be an ordinary citizen, never always-dominant
print(f"w[:,0] mean {weights[:, 0].mean():.4f} | max {weights[:, 0].max():.4f}  (1/K = {1/K:.4f})")

## Accuracy metric (blank 6 — ratify after looking)

Join by the step clock. **Convention fix:** the executed slice starts at `s0 + STEPS_PER_KNOT`,
not `s0` — knot 0 is the state one knot *after* plan birth (end-of-knot capture), so the
skeleton's `s0`-anchored slice carried a spurious one-knot offset that inflates the calm baseline.

In [ ]:
def plan_error(plan_rH3, summary=np.mean):
    """Per-replan deviation between a plan and what was actually executed over its window.
    summary: np.mean (provisional blank-6 choice) | np.max | lambda d: d[-1] (terminal)."""
    errs = np.full(R, np.nan)
    for k in range(R):
        s0 = replan_steps[k]
        seg = exec_path[s0 + STEPS_PER_KNOT : s0 + (H + 1) * STEPS_PER_KNOT : STEPS_PER_KNOT]
        m = min(len(seg), H)
        if m:
            dev = np.linalg.norm(plan_rH3[k, :m] - seg[:m], axis=1)
            errs[k] = summary(dev)
    return errs

dev_c = plan_error(cp)
dev_m = plan_error(wm)
pre_grasp = dev_c[replan_steps * DT < 11.5]
print(f"committed-plan deviation: median {np.nanmedian(dev_c)*1000:.1f} mm | "
      f"pre-grasp median {np.nanmedian(pre_grasp)*1000:.1f} mm | "
      f"max {np.nanmax(dev_c)*1000:.0f} mm at t={replan_steps[np.nanargmax(dev_c)]*DT:.1f} s")
print(f"provisional threshold (3x pre-grasp median): {3*np.nanmedian(pre_grasp)*1000:.1f} mm  <- DECISION(Jerry)")

In [ ]:
# ---- the figure ----
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(12, 10.5), sharex=True)
STRIDE = 12          # plot every STRIDE-th replan to declutter

for k in range(0, R, STRIDE):
    ax1.plot(plan_t[k], ef[k, :, :, 0], "-", color="tab:orange", alpha=0.10, lw=0.6)
    ax1.plot(plan_t[k], cp[k, :, 0], "-", color="tab:red", alpha=0.45, lw=0.9)
    ax1.plot(plan_t[k], wm[k, :, 0], "-", color="tab:blue", alpha=0.45, lw=0.9)
ax1.plot(exec_t, exec_path[:, 0], "k-", lw=2.0, label="executed")
ax1.plot([], [], color="tab:red", label="committed plan (pinned U_nom)")
ax1.plot([], [], color="tab:blue", label="weighted-mean path")
ax1.plot([], [], color="tab:orange", alpha=0.6, label="elite fan (top 10)")
ax1.set_ylabel("EE x (m)"); ax1.legend(fontsize=8, loc="lower right")
ax1.set_title("Plan vs executed — separated episode, unknown/L1-on, seed 1 (windup on tape)",
              fontsize=11, loc="left")

for k in range(0, R, STRIDE):
    ax2.plot(plan_t[k], ef[k, :, :, 2], "-", color="tab:orange", alpha=0.10, lw=0.6)
    ax2.plot(plan_t[k], cp[k, :, 2], "-", color="tab:red", alpha=0.45, lw=0.9)
ax2.plot(exec_t, exec_path[:, 2], "k-", lw=2.0, label="executed")
ax2.set_ylabel("EE z (m)"); ax2.legend(fontsize=8, loc="lower right")

ax3.plot(replan_steps * DT, dev_c, color="tab:red", lw=1.1, label="committed-plan deviation")
ax3.plot(replan_steps * DT, dev_m, color="tab:blue", lw=1.1, alpha=0.8, label="weighted-mean deviation")
ax3.axhline(3 * np.nanmedian(pre_grasp), color="0.3", ls="--", lw=1,
            label="provisional threshold (3x calm median)")
ax3.set_ylabel("plan deviation (m)"); ax3.set_xlabel("time (s)")
ax3.legend(fontsize=8); ax3.set_ylim(bottom=0)

for ax in (ax1, ax2, ax3):
    ax.grid(alpha=0.3)
    for tv in (12, 17):
        ax.axvline(tv, color="0.4", ls=":", lw=1)

plt.tight_layout()
out_png = os.path.basename(npz_file_path).replace(".npz", "_analysis.png")
plt.savefig(out_png, dpi=150)
print("wrote", out_png)
plt.show()

## Reading the figure — and the two decisions

**What the panels show:** pre-grasp, committed plan and weighted mean hug the executed line
(~5–6 mm median deviation over a 0.32 s horizon — the planner's imagination is honest when the
model is right). The elite fan **widens at events** — the uncertainty signature matching the ESS
story. Deviation spikes to ~60+ mm between grasp and switch (the adaptation transient invalidates
in-flight plans) and goes structurally odd after t≈24 where the windup held j3 at its limit —
plans made against a clamped joint cannot be executed. That segment is the CBF-chapter motivation
rendered as data.

**DECISION (blank 5) — which candidate is "the plan"?** Recommendation to ratify or overrule:
the **committed plan** (dynamically real, singular, what the controller commits to), with the
elite fan as the uncertainty band behind it; demote the weighted mean (fictional as a trajectory,
and the panels show it adds little).

**DECISION (blank 6) — the metric.** Provisional: *mean deviation over the horizon, per replan*,
flagged when above **3× the calm-phase median** (≈17 mm on this tape). Ratify the summary
statistic and the multiplier, or replace.

**Next (fresh conversation):** the polished thesis figure + the animation (time-slider over
replans) from this same npz, once the two decisions are stamped.